# Difuzija i utjecaj – primjer

**Poglavlje:** [8. Difuzija inovacija i utjecaj](../content/08_difuzija_inovacija_utjecaj.md)

Jednostavan graf ilustrira kako se **različite mjere centralnosti** ne slažu: čvor koji je „most” između dviju skupina nije nužno onaj s najviše direktnih veza. U kulturnim studijima takve razlike čitamo kao **nejednaku vidljivost** i **različite uloge u cirkulaciji** (npr. broker između scena vs. popularan račun unutar jedne zajednice). Brojevi su **heuristika**, ne dokaz moralnog ili stvarnog „utjecaja” na mišljenje.

Dalje u bilježnici: **Plotly** — animacija valova na malom `G`, **SI** s Monte Carlo krivuljom, usporedba izvora, histogram koraka, zatim **bonus**: Zachary „karate club”, stupčasti/kumulativni pregled, heatmapa više SI-traka, usporedba izvora 0 vs. 33, **Sankey** (sintaksa), te **izvoz HTML** u mapu `exports/`.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

# Mali graf: "jezgra" (1–2–3–4) i "rep" (5) spojen samo preko 4
G = nx.Graph()
G.add_edges_from([(1, 2), (1, 3), (2, 3), (2, 4), (3, 4), (4, 5)])

deg = dict(G.degree())
bet = nx.betweenness_centrality(G)
eig = nx.eigenvector_centrality(G, max_iter=500)

print("Degree (direktne veze):", deg)
print('Betweenness ("mostovi" kroz čvor):', {k: round(v, 3) for k, v in bet.items()})
print("Eigenvector (povezanost s utjecajnijim susjedima):", {k: round(v, 3) for k, v in eig.items()})

**Kulturno čitanje (ne matematika sama za sebe).** Visok **degree** može odgovarati „glasnoj” poziciji unutar grupe. Visok **betweenness** sugerira **brokersku** ulogu: tko drži vezu prema drugačijem dijelu mreže može usmjeravati što „prelazi” iz jedne scene u drugu. **Eigenvector** nagrađuje one koji su povezani s drugim dobro povezanim čvorovima — analogija: legitimitet kroz povezanost s već etabliranim akterima. Nijedna mjera sama ne govori *što* se širi niti *kako* publika dekodira poruku.

In [ ]:
df = pd.DataFrame({"degree": deg, "betweenness": bet, "eigenvector": eig})
df = df.round(3)
display(df)

# Čvor 4 često ima visok betweenness jer je jedini most prema 5
max_b = max(bet, key=bet.get)
print(f"Najveći betweenness: čvor {max_b}")

In [ ]:
# Vizualizacija: tamnija boja = veći betweenness (proxy za "most")
pos = nx.spring_layout(G, seed=42)
node_color = [bet[n] for n in G.nodes()]
nx.draw_networkx(
    G,
    pos,
    with_labels=True,
    node_color=node_color,
    cmap=plt.cm.Blues,
    edgecolors="black",
    font_size=11,
    node_size=800,
)
plt.title("Graf: boja čvora ∝ betweenness centrality")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
for n in sorted(G.nodes()):
    ego = nx.ego_graph(G, n, radius=2)
    print(f"Čvor {n}: doseg (|ego|, r=2) = {ego.number_of_nodes()}")

## Širenje informacije: deterministički valovi i stohastički SI

**Deterministički val** (sirina-prvo): u svakom koraku svi *neposredni* susjedi tek obaviještenih postaju obaviješteni — kao da se poruka širi sinkrono, bez kašnjenja na pojedinačnoj vezi. To je **najkraći** put od izvora u diskretnom vremenu.

**Stohastički SI** (jednostavna „zaraza”): u svakom koraku svaka veza od obaviještenog prema neobaviještenom s vjerojatnošću `p` prenosi informaciju. Jedan korak = jedan „takt” simultanih pokušaja. To daje **varijabilnost** puta i brzine širenja — blisko heuristici „viralnosti”, ali bez tvrdnje o stvarnoj kauzalnosti ili sadržaju poruke.

Ispod: **Plotly** — (1) interaktivna animacija valova na istom `G`, (2) prosjek ± std broja obaviještenih kroz korake, (3) usporedba izvora čvor **1** (jezgra) vs **5** (periferija), (4) distribucija broja koraka do potpunog obavještavanja.


In [ ]:
import numpy as np
import plotly.graph_objects as go

def _SHOW(fig):
    try:
        fig.show()
    except Exception:
        pass


def valovi_informacije(G, izvor):
    """Deterministicki BFS-valovi: lista skupova kumulativno obavijestenih nakon svakog vala."""
    obavijesteni = {izvor}
    granica = {izvor}
    snimke = [set(obavijesteni)]
    while True:
        sljedeci = set()
        for u in granica:
            for v in G[u]:
                if v not in obavijesteni:
                    sljedeci.add(v)
        if not sljedeci:
            break
        obavijesteni |= sljedeci
        snimke.append(set(obavijesteni))
        granica = sljedeci
    return snimke


def si_jedan_korak(G, obavijesteni, p, rng):
    """Stohasticki SI: simultani pokušaji preko svih veza iz obavijestenih."""
    novi = set(obavijesteni)
    for u in obavijesteni:
        for v in G[u]:
            if v not in obavijesteni and rng.random() < p:
                novi.add(v)
    return novi


def si_put(G, izvor, p, rng, max_koraka=60):
    """Vraca povijest len(obavijesteni) po koracima do zasićenja ili max_koraka."""
    o = {izvor}
    pov = [len(o)]
    for _ in range(max_koraka):
        if len(o) >= G.number_of_nodes():
            break
        n = si_jedan_korak(G, o, p, rng)
        if n == o:
            pov.append(len(o))
            break
        o = n
        pov.append(len(o))
    return pov


def monte_karlo_krivulja(G, izvor, p, br_sim=80, seed=42, max_koraka=60):
    """Prosijecna i std. dev. krivulja broja obavijestenih kroz korake (padding zadnjom vrijednosti)."""
    rng_master = np.random.default_rng(seed)
    trake = []
    for i in range(br_sim):
        r = np.random.default_rng(int(rng_master.integers(0, 2**31 - 1)))
        trake.append(si_put(G, izvor, p, r, max_koraka))
    L = max(len(t) for t in trake)
    mat = np.full((br_sim, L), np.nan)
    for i, t in enumerate(trake):
        arr = np.array(t, dtype=float)
        mat[i, : len(t)] = arr
        if len(t) < L:
            mat[i, len(t) :] = arr[-1]
    return np.nanmean(mat, axis=0), np.nanstd(mat, axis=0)


def koraci_do_pune_mreze(G, izvor, p, br_sim=120, seed=7, max_koraka=120):
    """Broj koraka dok svi nisu obavijesteni (ili max); za histogram."""
    out = []
    rm = np.random.default_rng(seed)
    for i in range(br_sim):
        r = np.random.default_rng(int(rm.integers(0, 2**31 - 1)))
        o = {izvor}
        korak = 0
        while len(o) < G.number_of_nodes() and korak < max_koraka:
            n = si_jedan_korak(G, o, p, r)
            if n == o:
                break
            o = n
            korak += 1
        out.append(korak if len(o) >= G.number_of_nodes() else np.nan)
    return np.array(out, dtype=float)


def plotly_animacija_valova(G, pos, snimke, naslov="Valovi informacije (deterministicki model)"):
    node_list = list(G.nodes())
    xs = [float(pos[n][0]) for n in node_list]
    ys = [float(pos[n][1]) for n in node_list]
    ex, ey = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ex += [float(x0), float(x1), None]
        ey += [float(y0), float(y1), None]
    edge_trace = go.Scatter(
        x=ex,
        y=ey,
        mode="lines",
        line=dict(color="#b2bec3", width=2),
        hoverinfo="skip",
        showlegend=False,
    )

    def cvorovi_trace(snap):
        boje = ["#d63031" if n in snap else "#dfe6e9" for n in node_list]
        return go.Scatter(
            x=xs,
            y=ys,
            mode="markers+text",
            text=[str(n) for n in node_list],
            textposition="top center",
            textfont=dict(size=12),
            marker=dict(size=40, color=boje, line=dict(color="#2d3436", width=1.2)),
            hovertext=[f"čvor {n}" + (" — obaviješten" if n in snap else "") for n in node_list],
            hoverinfo="text",
            showlegend=False,
        )

    frames = []
    for i, snap in enumerate(snimke):
        frames.append(
            go.Frame(
                name=str(i),
                data=[edge_trace, cvorovi_trace(snap)],
                traces=[0, 1],
            )
        )

    fig = go.Figure(
        data=[edge_trace, cvorovi_trace(snimke[0])],
        frames=frames,
        layout=dict(
            title=naslov,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="y", scaleratio=1),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor="white",
            width=700,
            height=560,
            margin=dict(l=20, r=20, t=60, b=80),
        ),
    )
    slider_steps = []
    for i in range(len(frames)):
        slider_steps.append(
            dict(
                method="animate",
                args=[
                    [str(i)],
                    dict(mode="immediate", frame=dict(duration=450, redraw=True), transition=dict(duration=0)),
                ],
                label=f"{i}",
            )
        )
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                y=1.14,
                x=0,
                buttons=[
                    dict(
                        label="Play",
                        method="animate",
                        args=[
                            None,
                            dict(frame=dict(duration=500, redraw=True), fromcurrent=True, transition=dict(duration=0)),
                        ],
                    ),
                    dict(label="Pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False), transition=dict(duration=0))]),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                steps=slider_steps,
                x=0.05,
                len=0.9,
                y=-0.02,
                currentvalue=dict(prefix="Korak vala: "),
                pad=dict(t=30, b=10),
            )
        ],
    )
    return fig


In [ ]:
# Pozicije i snimci deterministickog sirenja od čvora 1
pos = nx.spring_layout(G, seed=42)
snimke = valovi_informacije(G, 1)
print("Broj koraka (valova):", len(snimke) - 1)
for i, s in enumerate(snimke):
    print(f"  k={i}: obaviješteni = {sorted(s)}")

fig_valovi = plotly_animacija_valova(G, pos, snimke)
_SHOW(fig_valovi)


In [ ]:
# Stohasticka krivulja: prosjek broja obaviještenih kroz korake (izvor = 1; veci p = brza saturacija)
p_si = 0.78
mu, sig = monte_karlo_krivulja(G, 1, p_si, br_sim=100, seed=11)
koraci = np.arange(len(mu))

fig_si = go.Figure()
fig_si.add_trace(
    go.Scatter(
        x=koraci,
        y=mu,
        mode="lines",
        name="prosjek",
        line=dict(color="#0984e3", width=2),
    )
)
fig_si.add_trace(
    go.Scatter(
        x=np.r_[koraci, koraci[::-1]],
        y=np.r_[mu + sig, (mu - sig)[::-1]],
        fill="toself",
        fillcolor="rgba(9, 132, 227, 0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        name="±1 std",
        hoverinfo="skip",
    )
)
fig_si.update_layout(
    title=f"SI model (p={p_si}), izvor čvor 1 — broj obaviještenih kroz korake (100 simulacija)",
    xaxis_title="korak",
    yaxis_title="broj obaviještenih čvorova",
    width=760,
    height=420,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
_SHOW(fig_si)


In [ ]:
# Usporedba izvora: jezgra (1) vs periferija (5) — ista vjerojatnost p
mu1, s1 = monte_karlo_krivulja(G, 1, p_si, br_sim=100, seed=21)
mu5, s5 = monte_karlo_krivulja(G, 5, p_si, br_sim=100, seed=22)
k1 = np.arange(len(mu1))
k5 = np.arange(len(mu5))

fig_cmp = go.Figure()
fig_cmp.add_trace(go.Scatter(x=k1, y=mu1, mode="lines", name="izvor 1 (jezgra)", line=dict(color="#d63031", width=2)))
fig_cmp.add_trace(go.Scatter(x=k5, y=mu5, mode="lines", name="izvor 5 (rep)", line=dict(color="#6c5ce7", width=2)))
fig_cmp.update_layout(
    title=f"SI (p={p_si}): gdje informacija „kreće” mijenja oblik krivulje",
    xaxis_title="korak",
    yaxis_title="prosječan broj obaviještenih",
    width=760,
    height=420,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
_SHOW(fig_cmp)


In [ ]:
# Distribucija: koliko koraka treba da svi budu obaviješteni (SI, izvor 1)
koraci_hist = koraci_do_pune_mreze(G, 1, p_si, br_sim=150, seed=99)
koraci_hist = koraci_hist[np.isfinite(koraci_hist)]

fig_hist = go.Figure(
    data=[
        go.Histogram(
            x=koraci_hist,
            nbinsx=min(25, max(5, int(len(np.unique(koraci_hist))) + 2)),
            marker=dict(color="#00b894", line=dict(color="#fff", width=0.5)),
        )
    ]
)
fig_hist.update_layout(
    title="Broj koraka do potpunog obavještavanja (SI, izvor 1)",
    xaxis_title="koraka",
    yaxis_title="frekvencija (simulacija)",
    width=760,
    height=380,
    template="plotly_white",
    bargap=0.05,
)
_SHOW(fig_hist)
print("Medijan koraka:", float(np.median(koraci_hist)))


**Kulturno čitanje simulacije.** Deterministički val je „čista” geometrija mreže: pokazuje **najmanji broj diskretnih koraka** do svih čvorova od izabranog izvora. Stohastički SI dodaje **slučajnost** koja može stajati uz šum u pažnji, algoritamsku randomizaciju feeda ili neujednačenu spremnost ljudi da „prenesu” dalje — u bilježnici to ostaje apstraktan parametar `p`, ali u istraživanju ga treba vezati za **kontekst i metode prikupljanja**. Usporedba izvora 1 i 5 ilustrira da **položaj u topologiji** mijenja dinamiku čak i uz isti `p`; to je heuristika za raspravu o **periferiji i jezgrama** u kulturnim poljima, ne dokaz o „jačem utjecaju” pojedinaca.

## Bonus: Zachary „karate club”, više grafikona i izvoz HTML

**Zacharyjev klub** (`nx.karate_club_graph`) klasičan je mali **empirijski** graf (34 čvora): često se koristi u SNA udžbenicima. Ovdje ga koristimo samo kao **ilustrativnu** mrežu veće gustine — ne tvrdimo identitet s jednim stvarnim klubom iz 1970-ih.

- **Mreža u boji** (jedan Plotly prikaz): čvor obojan prema **prvom valu** u kojem postaje obaviješten — brže od animacije na 34 čvora, ali isti model valova.
- **Stupčasti + linijski** pregled: koliko je *novih* čvorova po valu vs. kumulativno.
- **Heatmapa** više SI-traka (što više „šuma” u stazi, to tamnije varijacije).
- **Usporedba izvora** 0 vs. 33 na istom `K` i istom `p_si`.
- **Sankey** — mali **uravnotežen** lanac vrijednosti (pedagoški primjer sintakse, ne empirijski model).
- **`write_html`** — spremanje zadnje figure u mapu `exports/` (možeš otvoriti u pregledniku bez Jupytera).


In [ ]:
# Zacharyjev graf: valovi informacije od čvora 0
K = nx.karate_club_graph()
snaps_k = valovi_informacije(K, 0)
pos_k = nx.spring_layout(K, seed=7, k=0.35, iterations=50)
print("Zachary: |V| =", K.number_of_nodes(), ", valova do potpunog obuhvata =", len(snaps_k) - 1)
for i in (0, 1, 2, len(snaps_k) - 1):
    if i < len(snaps_k):
        print(f"  k={i}: |obaviješteni| = {len(snaps_k[i])}")

prvi_val = {}
for t, Sk in enumerate(snaps_k):
    for n in Sk:
        prvi_val.setdefault(n, t)
nodes_k = list(K.nodes())
boje_val = [prvi_val[n] for n in nodes_k]
xs_k = [float(pos_k[n][0]) for n in nodes_k]
ys_k = [float(pos_k[n][1]) for n in nodes_k]
ex_k, ey_k = [], []
for u, v in K.edges():
    x0, y0 = pos_k[u]
    x1, y1 = pos_k[v]
    ex_k += [float(x0), float(x1), None]
    ey_k += [float(y0), float(y1), None]
tr_k_e = go.Scatter(
    x=ex_k,
    y=ey_k,
    mode="lines",
    line=dict(color="#b2bec3", width=1),
    hoverinfo="skip",
    showlegend=False,
)
tr_k_n = go.Scatter(
    x=xs_k,
    y=ys_k,
    mode="markers+text",
    text=[str(n) for n in nodes_k],
    textposition="top center",
    textfont=dict(size=8),
    marker=dict(
        size=14,
        color=boje_val,
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="val k"),
        line=dict(color="#2d3436", width=0.6),
    ),
    hovertext=[f"čvor {n}, prvi val k={prvi_val[n]}" for n in nodes_k],
    hoverinfo="text",
    showlegend=False,
)
fig_karate_valovi = go.Figure(data=[tr_k_e, tr_k_n])
fig_karate_valovi.update_layout(
    title="Zachary karate club — prvi val obavještavanja po čvoru (izvor 0)",
    width=840,
    height=640,
    template="plotly_white",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="y", scaleratio=1),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    margin=dict(l=20, r=30, t=60, b=20),
)
_SHOW(fig_karate_valovi)


In [ ]:
# Novi po valu vs. kumulativno (Karate)
novi_po_valu = [len(snaps_k[0])]
for i in range(1, len(snaps_k)):
    novi_po_valu.append(len(snaps_k[i] - snaps_k[i - 1]))
kumulativno = [len(s) for s in snaps_k]
korak_lab = [f"k={i}" for i in range(len(snaps_k))]

fig_karate_bars = go.Figure()
fig_karate_bars.add_trace(
    go.Bar(x=korak_lab, y=novi_po_valu, name="novi u valu", marker_color="#6c5ce7")
)
fig_karate_bars.add_trace(
    go.Scatter(
        x=korak_lab,
        y=kumulativno,
        mode="lines+markers",
        name="kumulativno obaviješteni",
        yaxis="y2",
        line=dict(color="#d63031", width=2),
    )
)
fig_karate_bars.update_layout(
    title="Karate: koliko novih čvorova po valu vs. kumulativno",
    template="plotly_white",
    width=820,
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    yaxis=dict(title="novi u valu", side="left"),
    yaxis2=dict(title="ukupno obaviješteni", overlaying="y", side="right", showgrid=False),
)
_SHOW(fig_karate_bars)


In [ ]:
# Heatmapa: 16 SI simulacija na Karate (izvor 0, p = p_si)
p_k = p_si
n_traka = 16
mat = []
for t in range(n_traka):
    pov = si_put(K, 0, p_k, np.random.default_rng(5000 + t), max_koraka=35)
    mat.append(pov)
L = max(len(r) for r in mat)
Z = np.full((n_traka, L), np.nan)
for i, r in enumerate(mat):
    Z[i, : len(r)] = r
    if len(r) < L:
        Z[i, len(r) :] = r[-1]

fig_heat = go.Figure(
    data=go.Heatmap(
        z=Z,
        colorscale="Viridis",
        colorbar=dict(title="obaviješteni"),
        hovertemplate="sim=%{y} korak=%{x} broj=%{z}<extra></extra>",
    )
)
fig_heat.update_layout(
    title=f"SI na Karate: {n_traka} traka (p={p_k}, izvor 0)",
    xaxis_title="korak",
    yaxis_title="simulacija (indeks)",
    width=900,
    height=420,
    template="plotly_white",
)
_SHOW(fig_heat)


In [ ]:
# Usporedba izvora na Karate: čvor 0 ("Mr Hi") vs. 33 ("Officer")
mu_k0, _ = monte_karlo_krivulja(K, 0, p_k, br_sim=24, seed=101)
mu_k33, _ = monte_karlo_krivulja(K, 33, p_k, br_sim=24, seed=102)
x0 = np.arange(len(mu_k0))
x33 = np.arange(len(mu_k33))

fig_k_cmp = go.Figure()
fig_k_cmp.add_trace(go.Scatter(x=x0, y=mu_k0, mode="lines", name="izvor 0", line=dict(color="#0984e3", width=2)))
fig_k_cmp.add_trace(go.Scatter(x=x33, y=mu_k33, mode="lines", name="izvor 33", line=dict(color="#e17055", width=2)))
fig_k_cmp.update_layout(
    title=f"Karate: SI prosjek (p={p_k}) — dva tipična izvora",
    xaxis_title="korak",
    yaxis_title="prosječan broj obaviještenih",
    width=820,
    height=400,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
_SHOW(fig_k_cmp)


In [ ]:
# Sankey (uravnotežen lanac — ilustracija sintakse Plotlyja)
# Tok: A->B (10), B->C (10), C->D (10) — konzervacija na čvoru
fig_sankey = go.Figure(
    data=[
        go.Sankey(
            arrangement="snap",
            node=dict(
                pad=18,
                thickness=16,
                line=dict(color="black", width=0.4),
                label=["A (ulaz)", "B", "C", "D (izlaz)"],
                color=["#74b9ff", "#a29bfe", "#fd79a8", "#55efc4"],
            ),
            link=dict(source=[0, 1, 2], target=[1, 2, 3], value=[10, 10, 10]),
        )
    ]
)
fig_sankey.update_layout(title="Sankey: jednostavan uravnotežen tok (nije empirijski model)", width=760, height=420, font=dict(size=12))
_SHOW(fig_sankey)


In [ ]:
# Spremi zadnju interaktivnu figuru kao HTML (otvori u pregledniku)
from pathlib import Path

_root = Path.cwd()
export_dir = _root / "code" / "exports" if (_root / "code").is_dir() else _root / "exports"
export_dir.mkdir(parents=True, exist_ok=True)
html_path = export_dir / "08_plotly_bonus_karate_cmp.html"
fig_k_cmp.write_html(html_path, include_plotlyjs="cdn", full_html=True)
print("Spremljeno:", html_path.resolve())


**Napomena.** `write_html` koristi Plotly CDN za JavaScript — treba internetsku vezu pri prvom otvaranju datoteke. Za potpuno offline izvoz postoji argument `include_plotlyjs=True` (veća datoteka). Heatmapa i usporedbe ovise o `p_si` i seedovima; mijenjaj ih da vidiš kako „šum” raste ili pada.